# Large Emotional Model (LEM) - Demo Notebook

This notebook demonstrates the proof-of-concept for next-state prediction in consumer behavior modeling.


## 1. Data Generation

Generate synthetic consumer behavior data.


In [ ]:
import sys
sys.path.append('..')

from generate_data import ConsumerDataGenerator
import pandas as pd
import numpy as np

# Generate data
generator = ConsumerDataGenerator(n_consumers=5000, n_timesteps=100, seed=42)
events_df, states_hidden = generator.generate()

print(f"Generated {len(events_df)} events")
print(f"State shape: {states_hidden.shape}")
print("\nFirst few events:")
print(events_df.head())


## 2. Baseline Evaluation

Evaluate baseline models.


In [ ]:
from models.baselines import RandomBaseline, StaticPreferenceBaseline, evaluate_baseline

# Load data
events_df = pd.read_csv('../data/events.csv')

# Split validation set
user_ids = events_df['user_id'].unique()
np.random.seed(42)
np.random.shuffle(user_ids)
split_idx = int(0.8 * len(user_ids))
val_users = set(user_ids[split_idx:])
val_df = events_df[events_df['user_id'].isin(val_users)].copy()

# Generate base traits
n_consumers = len(events_df['user_id'].unique())
np.random.seed(42)
base_traits = np.random.uniform(0, 1, size=(n_consumers, 5))

# Evaluate baselines
categories = sorted(events_df['category'].unique())
brands = sorted([b for b in events_df['brand'].unique() if b != 'none'])

print("Evaluating Random Baseline...")
random_baseline = RandomBaseline(categories, brands)
random_metrics = evaluate_baseline(random_baseline, val_df)

print("\nEvaluating Static Preference Baseline...")
static_baseline = StaticPreferenceBaseline(categories, brands)
static_baseline.fit(val_df, base_traits)
static_metrics = evaluate_baseline(static_baseline, val_df)

print("\nBaseline Results:")
print(f"Random - Accuracy: {random_metrics['accuracy']:.4f}, NLL: {random_metrics['nll']:.4f}")
print(f"Static - Accuracy: {static_metrics['accuracy']:.4f}, NLL: {static_metrics['nll']:.4f}")


## 3. Model Evaluation

Compare LEM against baselines.


In [ ]:
# Load evaluation results
import json

try:
    with open('../eval/metrics.json', 'r') as f:
        results = json.load(f)
    
    print("Evaluation Results:")
    print("="*60)
    print(f"\nRandom Baseline:")
    print(f"  Accuracy: {results['random_baseline']['accuracy']:.4f}")
    print(f"  NLL: {results['random_baseline']['nll']:.4f}")
    print(f"  Entropy: {results['random_baseline']['entropy']:.4f}")
    
    print(f"\nStatic Baseline:")
    print(f"  Accuracy: {results['static_baseline']['accuracy']:.4f}")
    print(f"  NLL: {results['static_baseline']['nll']:.4f}")
    print(f"  Entropy: {results['static_baseline']['entropy']:.4f}")
    
    print(f"\nLEM:")
    print(f"  Accuracy: {results['lem']['accuracy']:.4f}")
    print(f"  NLL: {results['lem']['nll']:.4f}")
    print(f"  Entropy: {results['lem']['entropy']:.4f}")
    
    # Compute improvements
    acc_improvement = ((results['lem']['accuracy'] - results['static_baseline']['accuracy']) / results['static_baseline']['accuracy']) * 100
    nll_reduction = ((results['static_baseline']['nll'] - results['lem']['nll']) / results['static_baseline']['nll']) * 100
    entropy_reduction = ((results['static_baseline']['entropy'] - results['lem']['entropy']) / results['static_baseline']['entropy']) * 100
    
    print(f"\nImprovements:")
    print(f"  Accuracy: +{acc_improvement:.1f}%")
    print(f"  NLL: -{nll_reduction:.1f}%")
    print(f"  Entropy: -{entropy_reduction:.1f}%") 
except FileNotFoundError:
    print("Evaluation results not found. Please run eval.py first.")


## 4. Visualizations

Display generated plots.


In [ ]:
from IPython.display import Image, display
import os

plot_dir = '../plots'
plots = [
    'before_vs_after_accuracy.png',
    'entropy_comparison.png',
    'state_trajectories.png',
    'population_state_distribution.png',
    'behavioral_regime_shifts.png',
    'training_history.png'
]

for plot_file in plots:
    plot_path = os.path.join(plot_dir, plot_file)
    if os.path.exists(plot_path):
        print(f"\n{plot_file}:")
        display(Image(plot_path))
    else:
        print(f"{plot_file} not found")


## 5. Conclusion

Display final research conclusion.


In [ ]:
# Display conclusion
try:
    with open('../eval/conclusion.txt', 'r') as f:
        conclusion = f.read()
    print(conclusion)
except FileNotFoundError:
    print("Conclusion not found. Run generate_conclusion.py first.")
